In [ ]:
import pandas as pd

sms_db = pd.read_csv("./all_sms.csv")
sms_db['date'] = pd.to_datetime(sms_db['date'], unit='s')
sms_db.dtypes

id                     int64
date          datetime64[us]
sender                   str
text                     str
is_from_me              bool
service                  str
dtype: object

In [13]:
sms_db = sms_db[sms_db['sender'] != 'me']
sms_db

,id,date,sender,text,is_from_me,service
2,3,2021-05-21 07:07:00.000000,VD-HDFCBK,Your mobile number and device is successfully ...,False,SMS
3,4,2021-05-21 07:07:03.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
5,6,2021-05-21 07:09:37.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
7,8,2021-05-21 07:09:40.000000,VK-HDFCBK,Device SMS count Exceeded For The Day,False,SMS
10,11,2021-05-21 08:19:47.000000,VK-ICICIB,"Dear Customer, registration for UPI has starte...",False,SMS
...,...,...,...,...,...,...
12124,18204,2026-03-04 14:26:16.869241,TRAIND-G,"ವಂಚಕರ ವಿರುದ್ಧ ಸ್ಪ್ಯಾಮ್ ವರದಿ ಮಾಡಿ. ಆದರೆ, ಫೋನಿನ ...",False,SMS
12125,18212,2026-03-05 07:24:09.798212,TRAIND-G,स्पॅमरच्या विरोधात कारवाईसाठी स्पॅमचा अहवाल द्...,False,SMS
12126,18215,2026-03-05 08:04:54.587784,AIRTEL-S,Never respond to emails/embedded links in mess...,False,SMS
12127,18223,2026-03-05 22:34:08.041849,MAHABK-S,"Dear Customer, there is no transaction in your...",False,SMS


In [14]:
# Check whether `id` is sequential and identify gaps/duplicates
id_series = pd.to_numeric(sms_db["id"], errors="coerce").dropna().astype(int)

row_count = len(sms_db)
min_id = id_series.min()
max_id = id_series.max()
unique_ids = id_series.nunique()

is_exact_1_to_n = (min_id == 1) and (max_id == row_count) and (unique_ids == row_count)
is_contiguous_min_to_max = (unique_ids == (max_id - min_id + 1))

print(f"rows: {row_count}")
print(f"id min/max: {min_id}/{max_id}")
print(f"unique ids: {unique_ids}")
print(f"sequential 1..rows? {is_exact_1_to_n}")
print(f"contiguous min..max? {is_contiguous_min_to_max}")

# Missing IDs in the observed min..max range
expected_ids = pd.Index(range(min_id, max_id + 1))
missing_ids = expected_ids.difference(pd.Index(id_series.unique()))

print(f"missing id count: {len(missing_ids)}")
print("first 20 missing ids:", missing_ids[:20].tolist())

# Duplicate IDs (if any)
dup_counts = id_series.value_counts()
duplicate_ids = dup_counts[dup_counts > 1]
print(f"duplicate id count: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print("first 20 duplicate ids:", duplicate_ids.head(20).to_dict())

rows: 11984
id min/max: 3/18252
unique ids: 11984
sequential 1..rows? False
contiguous min..max? False
missing id count: 6266
first 20 missing ids: [5, 7, 9, 10, 14, 15, 26, 35, 36, 37, 38, 39, 40, 50, 60, 66, 79, 84, 99, 137]
duplicate id count: 0


In [15]:
sms_db['sender'].value_counts()

sender
VZ-611123          1007
VZ-612345          1001
VD-611126           256
AD-HDFCBK           242
JZ-JioPay           207
                   ... 
ADHAAR-S(smsft)       1
+918977520171         1
HDFCBN-S(smsft)       1
SBIBNK-S(smsft)       1
AIRTEL-S              1
Name: count, Length: 1648, dtype: int64

In [3]:
# Filter out sent messages
inbox_df = sms_db[sms_db['sender'] != 'me'].copy()

# Basic sender stats
unique_senders = inbox_df['sender'].nunique()
print(f"Total incoming messages: {len(inbox_df)}")
print(f"Total unique senders: {unique_senders}")

# Top 20 senders by volume
top_senders = inbox_df['sender'].value_counts().head(50)
print("\nTop 20 senders by message volume:")
print(top_senders)

Total incoming messages: 11984
Total unique senders: 1648

Top 20 senders by message volume:
sender
VZ-611123    1007
VZ-612345    1001
VD-611126     256
AD-HDFCBK     242
JZ-JioPay     207
VM-611121     192
VM-HDFCBK     163
VZ-EXPIRY     146
JK-620016     146
JD-620016     136
JM-620016     134
VD-HDFCBN     133
JZ-JIOINF     132
JZ-JIOPAY     125
JX-HDFCBK     121
JX-620016     119
VM-HDFCBN     116
VM-ViCARE     114
VD-HDFCBK     111
VM-611126     109
VM-IDFCFB     101
5512111       100
TX-SLCEIT      96
VZ-ViRoam      81
QP-CHGGIn      71
JZ-JioSvc      70
VK-NSETRA      66
VM-NSESMS      63
CP-OneCrd      62
AX-HDFCBK      59
BW-SBIUPI      58
AD-HDFCBN      57
VM-FLPKRT      56
TM-IDFCFB      56
5040406        55
TM-TATANU      55
JM-HDFCBN      54
JZ-JIOVOC      53
VM-adidas      52
50404          51
BH-CHGGIn      49
BA-SBIUPI      49
VD-IDFCFB      49
JD-SBIUPI      47
VM-BSELTD      45
AD-OneCrd      43
AD-IDFCFB      43
AD-SBIUPI      40
VM-NOBRKR      40
VK-MAHABK      39


In [4]:
import re

def categorize_sender(sender):
    sender = str(sender).strip()
    
    # 1. Regular Mobile Number: optional '+' followed by 10 to 15 digits
    if re.match(r'^\+?[0-9]{10,15}$', sender):
        return 'Mobile Number'
    
    # 2. Numeric Shortcode: purely numeric, less than 10 digits
    elif re.match(r'^[0-9]{3,9}$', sender):
        return 'Numeric Shortcode'
    
    # 3. Commercial/Brand: standard XX-YYYYYY Indian gateway format
    elif re.match(r'^[A-Za-z]{2}-.+$', sender):
        return 'Commercial/Brand (Prefixed)'
    
    # 4. Commercial/Brand: other textual names (e.g., 'JioPay', 'HDFCBK' without prefix)
    elif re.search(r'[A-Za-z]', sender):
        return 'Commercial/Brand (Textual)'
    
    # 5. Others: to catch edge cases
    else:
        return 'Others'

# Apply the categorization
inbox_df['sender_category'] = inbox_df['sender'].apply(categorize_sender)

# Get counts
category_counts = inbox_df['sender_category'].value_counts()
print("Message volume by Sender Category:")
print(category_counts, "\n")

# Let's inspect unique senders in other/edge-case categories
others_senders = inbox_df[inbox_df['sender_category'] == 'Others']['sender'].unique()
print(f"Unique senders in 'Others': {len(others_senders)}")
if len(others_senders) > 0:
    print(others_senders[:20])

Message volume by Sender Category:
sender_category
Commercial/Brand (Prefixed)    11259
Mobile Number                    325
Numeric Shortcode                268
Commercial/Brand (Textual)       132
Name: count, dtype: int64 

Unique senders in 'Others': 0


In [18]:
inbox_df[inbox_df['sender_category'] == 'Mobile Number']

,id,date,sender,text,is_from_me,service,sender_category
50,55,2021-05-29 10:00:11.000000,+919351197465,6-Layers Protection C0R0NA-virus.\nwear N95 Ma...,False,SMS,Mobile Number
94,102,2021-06-06 07:17:53.000000,+917703967847,"Dear 96991837XX,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
118,126,2021-06-09 10:02:53.000000,+918700855621,"Congratulations,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
266,269,2021-06-30 05:10:44.000000,+917990025887,Your 9699XXXX27 is selected for Rs 5250.00 dir...,False,SMS,Mobile Number
398,409,2021-07-26 07:06:58.000000,+918595950300,Your Citi Credit Card No. YCPTXXXXXXXX8795 has...,False,SMS,Mobile Number
...,...,...,...,...,...,...,...
12112,17882,2026-02-07 20:33:24.276616,+919981657790,😞,True,SMS,Mobile Number
12113,17883,2026-02-07 20:33:25.558564,+919981657790,😞,True,SMS,Mobile Number
12114,17884,2026-02-07 20:33:47.968934,+919981657790,Tum so hi rhi ho na,True,SMS,Mobile Number
12115,17885,2026-02-07 20:34:13.965093,+919981657790,Kitna piya tha itna,True,SMS,Mobile Number


In [5]:
import re
import pandas as pd
import sys

# 1. Focus only on Commercial and Shortcode messages (Drop personal mobile numbers)
commercial_df = inbox_df[inbox_df['sender_category'] != 'Mobile Number'].copy()

# ==========================================
# STAGE 1: AMOUNT REQUIREMENT 
# ==========================================
# Transactions almost strictly require a currency format + number. 
# We're making sure we capture varied spaces between currency like 'Rs. 500', 'Rs.500', 'INR500', '₹ 100'.
amount_pattern = re.compile(
    r'(?:rs\.?|inr|₹)\s*[\d,]+(?:\.\d{1,2})?|[\d,]+(?:\.\d{1,2})?\s*(?:rs\.?|inr|₹)', 
    re.IGNORECASE
)

def has_amount(text):
    if not isinstance(text, str):
        return False
    return bool(amount_pattern.search(text))

print("STAGE 1: Applying amount requirement filter...")
commercial_df['1_has_amount'] = commercial_df['text'].apply(has_amount)

# Analyze breakdown
txn_counts_1 = commercial_df['1_has_amount'].value_counts()
print(f"\nTotal Commercial/Shortcode messages: {len(commercial_df)}")
print("Messages with Amount vs Without (Likely Transaction vs Not):")
print(txn_counts_1)
print(f"Percentage retaining amount: {(txn_counts_1.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

# Sample match
print("--- SAMPLE MATCHED MESSAGES WITH AMOUNT (True) ---")
matched_sample = commercial_df[commercial_df['1_has_amount'] == True]['text'].dropna().sample(5, random_state=42)
for text in matched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}")

# Sample unmatch
print("\n--- SAMPLE FILTERED OUT AS NO AMOUNT (False) ---")
unmatched_sample = commercial_df[commercial_df['1_has_amount'] == False]['text'].dropna().sample(5, random_state=42)
for text in unmatched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}")

STAGE 1: Applying amount requirement filter...

Total Commercial/Shortcode messages: 11659
Messages with Amount vs Without (Likely Transaction vs Not):
1_has_amount
False    6480
True     5179
Name: count, dtype: int64
Percentage retaining amount: 44.42%

--- SAMPLE MATCHED MESSAGES WITH AMOUNT (True) ---
- Vi T20 Cricket Pack! Rs901=3GB/day + 48GB EXTRA + All India UL Calls valid for 70days + 1year Disney+ Hotstar Mobile. Recharge NOW! https://bit.ly/3io
- Rs500.0 debited@SBI UPI frm A/cX6254 on 26Aug22 RefNo 223838987367. If not done by u, fwd this SMS to 9223008333/Call 1800111109 or 09449112211 to blo
- Dear SBI User, your A/c X6254-debited by Rs838.0 on 07Jun22 transfer to Flipkart Ref No 215810058234. If not done by u, fwd this SMS to 9223008333/Cal
- Dear Manish,  Get more cash effortlessly with HDFC Bank Personal Loan up to Rs 5 lacs at Best rates. Paperless. Click: hdfcbk.net/vXKVm5vb  T&C
- Missing your favorite songs? Enjoy 3months Ad Free music in HD quality with 6GB for 15

In [ ]:
import os

# Create RESULTS directory
os.makedirs('RESULTS', exist_ok=True)

# Separate the datasets based on Stage 1
likely_transactions = commercial_df[commercial_df['1_has_amount'] == True]
non_transactions = commercial_df[commercial_df['1_has_amount'] == False]

# Save to a single Excel file with two sheets
output_path = 'RESULTS/1_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 6624 likely transactions and 5035 non-transactions to RESULTS/1_transactions_split.xlsx


In [ ]:
# ==========================================
# STAGE 2: REQUIRE MASKED ACCOUNT/CARD NUMBER
# ==========================================
# Real bank transaction SMSes almost always reference a masked account or card number.
# This is the strongest positive signal for identifying bank/card transactions
# and naturally excludes promos, wallets (Simpl, slice), brokerage EOD reports, etc.

acct_pattern = re.compile(
    r'a/?c\s*(?:no\.?\s*)?[X*x]+\d+|'          # A/c XX6254, A/C No XXXXXXX7902, a/c *9141
    r'a/?c\s*(?:no\.?\s*)?\*+\d+|'              # A/c **9141
    r'card\s*(?:no\.?\s*)?[Xx*]+\d+|'           # Card x6719, Card XX3782
    r'card\s+\d{4}\b|'                           # Card 0816
    r'card\s+ending\s+[Xx*]*\d+',               # card ending XX3782
    re.IGNORECASE
)

def has_account_or_card(row):
    text = row['text']
    if not isinstance(text, str):
        return False
    # MUST have passed Stage 1 (has amount)
    if not row['1_has_amount']:
        return False
    return bool(acct_pattern.search(text))

print('STAGE 2: Applying masked account/card number requirement...')
commercial_df['2_has_acct'] = commercial_df.apply(has_account_or_card, axis=1)

txn_counts_2 = commercial_df['2_has_acct'].value_counts()
print(f'\nMessages with account/card number vs without:')
print(txn_counts_2)
print(f'Percentage retained: {(txn_counts_2.get(True, 0) / len(commercial_df)) * 100:.2f}%\n')

print('--- SAMPLE MATCHED (has account/card number) ---')
matched = commercial_df[commercial_df['2_has_acct'] == True]['text'].dropna()
for text in matched.sample(min(5, len(matched)), random_state=42):
    print(f'- {text[:200].replace(chr(10), " ")}')

print('\n--- SAMPLE FILTERED OUT (had amount but no account/card) ---')
filtered = commercial_df[(commercial_df['1_has_amount'] == True) & (commercial_df['2_has_acct'] == False)]['text'].dropna()
for text in filtered.sample(min(5, len(filtered)), random_state=42):
    print(f'- {text[:200].replace(chr(10), " ")}')


In [ ]:
import os

# Separate the datasets based on Stage 2
likely_transactions = commercial_df[commercial_df['2_has_acct'] == True]
non_transactions = commercial_df[commercial_df['2_has_acct'] == False]

output_path = 'RESULTS/2_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f'Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}')


In [ ]:
# ==========================================
# STAGE 3: REQUIRE TRANSACTION ACTION VERB
# ==========================================
# Messages must contain a verb/phrase indicating an actual debit or credit event.
# This filters out promos that happen to mention an account number
# (e.g. 'Get EasyEMI on HDFC Bank Credit Card xx6719').

txn_verb_pattern = re.compile(
    r'\b(debited|credited|deducted|spent|paid|received|transferred|sent|reversed|refunded|used)\b|'
    r'\b(money\s+transfer|amt\s+sent|amt\s+received)\b',
    re.IGNORECASE
)

def has_txn_verb(row):
    text = row['text']
    if not isinstance(text, str):
        return False
    # MUST have passed Stage 2 (has amount + account/card)
    if not row['2_has_acct']:
        return False
    return bool(txn_verb_pattern.search(text))

print('STAGE 3: Applying transaction action verb requirement...')
commercial_df['3_has_verb'] = commercial_df.apply(has_txn_verb, axis=1)

txn_counts_3 = commercial_df['3_has_verb'].value_counts()
print(f'\nMessages with transaction verb vs without:')
print(txn_counts_3)
print(f'Percentage retained: {(txn_counts_3.get(True, 0) / len(commercial_df)) * 100:.2f}%\n')

print('--- SAMPLE MATCHED (has transaction verb) ---')
matched = commercial_df[commercial_df['3_has_verb'] == True]['text'].dropna()
for text in matched.sample(min(5, len(matched)), random_state=42):
    print(f'- {text[:200].replace(chr(10), " ")}')

print('\n--- SAMPLE FILTERED OUT (had account but no verb) ---')
filtered = commercial_df[(commercial_df['2_has_acct'] == True) & (commercial_df['3_has_verb'] == False)]['text'].dropna()
if len(filtered) > 0:
    for text in filtered.sample(min(5, len(filtered)), random_state=42):
        print(f'- {text[:200].replace(chr(10), " ")}')
else:
    print('No messages were filtered out in this stage.')


In [ ]:
import os

# Separate the datasets based on Stage 3
likely_transactions = commercial_df[commercial_df['3_has_verb'] == True]
non_transactions = commercial_df[commercial_df['3_has_verb'] == False]

output_path = 'RESULTS/3_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f'Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}')


In [ ]:
# ==========================================
# STAGE 4: OTP EXCLUSION (tightened)
# ==========================================
# Remove OTP/auth messages that may reference an account + verb.
# Tightened from original: dropped loose 'pin' and 'do not share' which
# caused false positives on legitimate transaction alerts.

otp_pattern = re.compile(
    r'\botp\b|\bone.?time.?password\b|\bverification.?code\b',
    re.IGNORECASE
)

def not_is_otp(row):
    text = row['text']
    if not isinstance(text, str):
        return False
    # MUST have passed Stage 3
    if not row['3_has_verb']:
        return False
    return not bool(otp_pattern.search(text))

print('STAGE 4: Applying OTP exclusion filter...')
commercial_df['4_not_otp'] = commercial_df.apply(not_is_otp, axis=1)

txn_counts_4 = commercial_df['4_not_otp'].value_counts()
print(f'\nAfter OTP exclusion:')
print(txn_counts_4)
print(f'Percentage retained: {(txn_counts_4.get(True, 0) / len(commercial_df)) * 100:.2f}%\n')

print('--- SAMPLE FILTERED OUT AS OTP ---')
filtered = commercial_df[(commercial_df['3_has_verb'] == True) & (commercial_df['4_not_otp'] == False)]['text'].dropna()
if len(filtered) > 0:
    for text in filtered.sample(min(5, len(filtered)), random_state=42):
        print(f'- {text[:200].replace(chr(10), " ")}')
else:
    print('No messages were filtered out in this stage.')


In [ ]:
import os

# Separate the datasets based on Stage 4
likely_transactions = commercial_df[commercial_df['4_not_otp'] == True]
non_transactions = commercial_df[commercial_df['4_not_otp'] == False]

output_path = 'RESULTS/4_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f'Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}')


In [ ]:
# ==========================================
# STAGE 5: COLLECT REQUEST EXCLUSION
# ==========================================
# Remove UPI collect/mandate requests — these are pending actions, not completed
# transactions. Only exclude 'request' in the context of payment requests,
# not 'request processed/completed' which indicates a done transaction.

collect_pattern = re.compile(
    r'has\s+requested\s+money|'
    r'requested\s+Rs\.?|'
    r'collect\s+request|'
    r'mandate\s+request|'
    r'request\s+from\s+you',
    re.IGNORECASE
)

def not_is_collect(row):
    text = row['text']
    if not isinstance(text, str):
        return False
    # MUST have passed Stage 4
    if not row['4_not_otp']:
        return False
    return not bool(collect_pattern.search(text))

print('STAGE 5: Applying collect/mandate request exclusion...')
commercial_df['5_not_collect'] = commercial_df.apply(not_is_collect, axis=1)

txn_counts_5 = commercial_df['5_not_collect'].value_counts()
print(f'\nAfter collect request exclusion:')
print(txn_counts_5)
print(f'Percentage retained: {(txn_counts_5.get(True, 0) / len(commercial_df)) * 100:.2f}%\n')

print('--- SAMPLE FILTERED OUT AS COLLECT REQUEST ---')
filtered = commercial_df[(commercial_df['4_not_otp'] == True) & (commercial_df['5_not_collect'] == False)]['text'].dropna()
if len(filtered) > 0:
    for text in filtered.sample(min(5, len(filtered)), random_state=42):
        print(f'- {text[:200].replace(chr(10), " ")}')
else:
    print('No messages were filtered out in this stage.')


In [ ]:
import os

# Separate the datasets based on final Stage 5
likely_transactions = commercial_df[commercial_df['5_not_collect'] == True]
non_transactions = commercial_df[commercial_df['5_not_collect'] == False]

output_path = 'RESULTS/5_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f'Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}')
print(f'\n--- PIPELINE SUMMARY ---')
print(f'Total commercial/shortcode messages: {len(commercial_df)}')
print(f'Stage 1 (has amount):        {commercial_df["1_has_amount"].sum()}')
print(f'Stage 2 (has account/card):   {commercial_df["2_has_acct"].sum()}')
print(f'Stage 3 (has txn verb):       {commercial_df["3_has_verb"].sum()}')
print(f'Stage 4 (not OTP):            {commercial_df["4_not_otp"].sum()}')
print(f'Stage 5 (not collect req):    {commercial_df["5_not_collect"].sum()}')
